# 🏋️ Flexa — Workout Split Classifier (Module 4)
## Multi-Class Classification: Predicting Optimal Workout Split from User Profile

**Dataset:** 436 survey responses  
**Features:** Age · Gender · Training Frequency · Experience Level · Primary Goal  
**Target:** Workout Split (Full Body / PPL / PPL x2 / Upper-Lower / Bro Split)  
**Models:** Random Forest · XGBoost (+ SMOTE oversampling + hyperparameter tuning)

---
> *This notebook is the FYP documentation artifact for Module 4 of the Flexa AI Fitness Planner.*

## 1. Import Required Libraries

In [ ]:
# ── Standard data science stack ──────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# ── XGBoost ───────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Imbalanced-learn (SMOTE) ──────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# ── Plot style ────────────────────────────────────────────────────────────────
plt.style.use("dark_background")
sns.set_palette("husl")

print("All libraries imported successfully ✓")

## 2. Load and Inspect the Dataset

The dataset is collected via Google Forms with 436 responses. Column positions are used because the headers contain long descriptive text.

In [ ]:
# ── Dataset path ─────────────────────────────────────────────────────────────
# Update this path if your dataset is in a different location.
import os, sys

DATASET_PATH = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "..", "flexa-frontend", "WorkoutPlanning_Dataset.xlsx"
)

# ── Load ──────────────────────────────────────────────────────────────────────
raw = pd.read_excel(DATASET_PATH)
print(f"Raw dataset shape: {raw.shape}")
print(f"\nColumn headers (positional):")
for i, col in enumerate(raw.columns):
    print(f"  [{i}] {col[:60].strip()}")

raw.head(3)

In [ ]:
# ── Basic inspection ─────────────────────────────────────────────────────────
print("Data types:\n", raw.dtypes.to_string())
print("\nMissing values:\n", raw.isnull().sum().to_string())
print("\nTarget class distribution:")
print(raw.iloc[:, 7].value_counts().to_string())

## 3. Data Cleaning and Preprocessing

Steps:
- Rename columns to short descriptive names using positional indexing
- Drop `Timestamp` and `Name` (non-predictive)
- Normalize string values (strip whitespace, fix mixed casing)
- Parse numeric frequency and age values
- Drop rows with missing target

In [ ]:
# ── Step 1: Rename columns by position ───────────────────────────────────────
df = raw.copy()
df.columns = ["timestamp", "name", "age", "gender", "frequency", "experience", "goal", "split"]

# ── Step 2: Drop irrelevant columns ──────────────────────────────────────────
# 'timestamp' and 'name' carry no predictive signal.
df.drop(columns=["timestamp", "name"], inplace=True)
print("Columns after drop:", list(df.columns))

# ── Step 3: Normalize string values ─────────────────────────────────────────
def norm_freq(v):
    """Extract the leading integer from strings like '4 days/week (Monday–Thursday)'."""
    v = str(v).strip()
    for d in ["2", "3", "4", "5", "6"]:
        if v.startswith(d):
            return int(d)
    return None  # will be dropped

def norm_exp(v):
    v = str(v).lower().strip()
    if "begin" in v: return "Beginner"
    if "inter" in v: return "Intermediate"
    if "advan" in v: return "Advanced"
    return None

def norm_goal(v):
    v = str(v).lower().strip()
    if "bulk"  in v: return "Bulking"
    if "cut"   in v: return "Cutting"
    if "main"  in v or "recomp" in v: return "Maintaining"
    return None

def norm_gender(v):
    v = str(v).lower().strip()
    if v.startswith("m"): return "Male"
    if v.startswith("f"): return "Female"
    return None

def norm_split(v):
    v = str(v).strip()
    if "Full Body"     in v: return "Full Body"
    if "PPL x2"        in v or "6 days" in v: return "PPL x2"
    if "Push/Pull/Legs" in v or "Push-Pull" in v: return "PPL"
    if "Upper/Lower"   in v or "Upper-Lower" in v: return "Upper/Lower"
    if "Bro Split"     in v: return "Bro Split"
    return None  # unknown — will be dropped

df["frequency"]  = df["frequency"].apply(norm_freq)
df["experience"] = df["experience"].apply(norm_exp)
df["goal"]       = df["goal"].apply(norm_goal)
df["gender"]     = df["gender"].apply(norm_gender)
df["split"]      = df["split"].apply(norm_split)
df["age"]        = pd.to_numeric(df["age"], errors="coerce")

# ── Step 4: Drop rows with any remaining nulls ────────────────────────────────
before = len(df)
df.dropna(inplace=True)
df = df.astype({"frequency": int, "age": float})
print(f"Rows kept after cleaning: {len(df)} / {before}")
print("\nClean sample:")
df.head()